## OpenAPI Tools with Foundry Agent Service  

### Installing Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Note: you may need to restart the kernel to use updated packages.


### Setting Up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
)
import jsonref

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

### Setting Up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Initializing the OpenAPI Tool

In [4]:
with open("./weather_openapi.json", "r") as f:
        openapi_weather = jsonref.loads(f.read())

# Initialize agent OpenApi tool using the read in OpenAPI spec
weather_tool = {
        "type": "openapi",
        "openapi":{
            "name": "weather",
            "spec": openapi_weather,
            "auth": {
                "type": "anonymous"
            },
        }
}

### Setting Up Our Agent with OpenAPI Tool Spec

In [5]:
agent_name = "weather-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful Weather Agent that provides weather information using the provided OpenAPI Tool",
        tools=[weather_tool]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: weather-agent:2, name: weather-agent, version: 2)


### Creating a Conversation Object for the Agent Chat System

In [6]:
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_2b334869be2dd2c700Da91V09A02DUhkCOTesE4BJuvg2lhpLY


In [7]:
user_input = "What's the weather like in New York City today?"

### Chat with The Agent

In [8]:
response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )
print(f"Weather Agent: {response.output_text}")

Weather Agent: **Current weather in New York City (as of the latest observation):**

- **Temperature**: 21°C (69°F)
- **Feels like**: 21°C (69°F)
- **Condition**: Light Rain
- **Cloud cover**: 100%
- **Humidity**: 84%
- **Wind**: 15 km/h (9 mph) from the North
- **Pressure**: 1013 hPa
- **Visibility**: 16 km (9 miles)
- **UV Index**: 1 (Low)

Light rain is falling with full cloud cover. It should stay mild throughout the day, with temperatures ranging from about 20–23°C (68–74°F). Expect continued patchy rain or drizzle at times, especially in the afternoon and evening, before clearing somewhat overnight.

For the rest of today (June 23, 2026), highs will reach 23°C (74°F) with a chance of more showers. If you need tomorrow's forecast or details for a specific time, just let me know!
